# 06 --- Conflict Resolution

**CCA Pattern**: The coordinator resolves contradictions using deterministic strategies:
1. Source reliability ranking
2. Majority consensus
3. Flag for human review

This is programmatic enforcement -- not LLM judgment.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.agent.conflict_resolver import resolve_conflict, RELIABILITY_SCORES
from research_agents.models.research import SourceReliability
from research_agents.data.sources import SOURCE_RELIABILITY_RATINGS

## How the Conflict Resolver Works

The `conflict_resolver.py` module uses **deterministic strategies** -- no LLM
reasoning at all. This is the CCA principle that programmatic enforcement beats
prompt-based guidance.

### The ConflictRecord Model

```python
class ConflictRecord(BaseModel):
    claim: str                 # The disputed factual claim
    sources_for: list[str]     # URLs supporting the claim
    sources_against: list[str] # URLs contradicting the claim
    resolution: str            # "majority", "highest_reliability", "flagged_for_human"
    confidence: float          # 0.0 to 1.0
    winning_side: str          # "for", "against", or "undecided"
```

`winning_side` matters: a record that only says *how* a conflict was resolved
is useless to the synthesis step, which needs to know *which way*.

### Reliability Scoring

Sources are scored by reliability tier:

| Tier | Score | Examples |
|------|-------|---------|
| HIGH | 3 | `.gov`, peer-reviewed journals, official statistics |
| MEDIUM | 2 | Established news outlets, `.edu`, consultancies |
| LOW | 1 | Blogs, unverified sources |
| UNKNOWN | 0 | Not yet assessed |

Ratings are keyed by domain (`"energy.gov"`) and sources arrive as full URLs;
the resolver matches them the same way the knowledge base does, so
`SOURCE_RELIABILITY_RATINGS` can be passed straight in.

### Three-Tier Resolution Strategy

1. **Reliability ranking**: Compare the *best* tier on each side. The side
with the more reliable source wins; three blogs never outrank one `.gov`.
Confidence scales with the tier gap: `0.5 + gap * 0.15`, capped at 0.95.
2. **Majority consensus**: If the best tiers are equal, the side with more
sources wins. Confidence = majority_size / total, capped at 0.85.
3. **Human review**: If both tier and count tie, flag for human review
with low confidence (0.3).

## Anti-Pattern: First Result Wins

The distractor on the exam is "take whichever source came back first". It
is simple and it *feels* decisive. It is also a function of arrival order,
which is a function of network latency, which has nothing to do with truth.
The cell below shows the verdict flipping when the same three reports arrive
in a different order.

In [ ]:
GOV = 'https://bls.gov/remote-work-stats'
MCKINSEY = 'https://mckinsey.com/future-of-work'
BLOG = 'https://workfromhome-blog.example.com/productivity'
CLAIM = 'Remote workers are more productive'

def domain(url: str) -> str:
    return url.split('/')[2]

def first_result_wins(reports: list[tuple[str, str]]) -> str:
    """ANTI-PATTERN: whichever source reported first decides the claim."""
    source, stance = reports[0]
    print(f'  (decided by {domain(source)})')
    return stance

blog_first = [(BLOG, 'against'), (MCKINSEY, 'for'), (GOV, 'for')]
gov_first = list(reversed(blog_first))

anti_a = first_result_wins(blog_first)
print(f'Blog reports first -> verdict: {anti_a}')
anti_b = first_result_wins(gov_first)
print(f'.gov reports first -> verdict: {anti_b}')
print(f'Same evidence, same verdict? {anti_a == anti_b}')

## Correct Pattern: Deterministic Resolution

`resolve_conflict()` takes the evidence sorted by stance, not by arrival, and
scores it by reliability. The same three reports give the same verdict in
either order.

In [ ]:
def deterministic_verdict(reports: list[tuple[str, str]]) -> tuple[str, object]:
    sources_for = [s for s, stance in reports if stance == 'for']
    sources_against = [s for s, stance in reports if stance == 'against']
    record = resolve_conflict(CLAIM, sources_for, sources_against, SOURCE_RELIABILITY_RATINGS)
    return record.winning_side, record

correct_a, record_a = deterministic_verdict(blog_first)
correct_b, record_b = deterministic_verdict(gov_first)
print(f'Blog reports first -> verdict: {correct_a}  ({record_a.resolution}, {record_a.confidence:.2f})')
print(f'.gov reports first -> verdict: {correct_b}  ({record_b.resolution}, {record_b.confidence:.2f})')
print(f'Same evidence, same verdict? {correct_a == correct_b}')

## Strategy 1: Source Reliability Ranking

In [ ]:
# Show the scoring constants
print('Reliability scores:')
for tier, score in RELIABILITY_SCORES.items():
    print(f'  {tier.value:<10s} = {score}')
print()

# High-reliability source beats low-reliability (ratings keyed by domain)
record = resolve_conflict(
    claim='Renewable energy accounts for 30% of global electricity',
    sources_for=['https://energy.gov/renewable-2024'],
    sources_against=['https://energyblog.example.com/renewables'],
    reliability_lookup=SOURCE_RELIABILITY_RATINGS,
)
print(f'Claim: {record.claim}')
print(f'Resolution: {record.resolution}')
print(f'Winning side: {record.winning_side}')
print(f'Confidence: {record.confidence:.2f}  # HIGH(3) vs LOW(1): gap 2 -> 0.5 + 2 * 0.15')

## Strategy 2: Majority Vote (when reliabilities tie)

If the best tier is the same on both sides, the resolver falls through to a
majority vote. This matters in the most common real-world case: several
moderately-reliable news outlets disagreeing with each other. No one side has
a reliability edge, but three sources saying the same thing is stronger
evidence than one source saying the opposite.

The confidence on a majority resolution is `majority_count / total_count`.
So 3 of 4 sources agreeing yields `0.75`. This is deliberately lower than
the confidence ceiling on reliability wins (`0.95`) -- a reliability-based
answer is treated as stronger evidence than a count-based answer, which
is what you'd want on the exam.

In [ ]:
# Three medium-reliability sources for the claim, one against
reliability_majority = {
    'https://reuters.com/article-a': SourceReliability.MEDIUM,
    'https://bbc.com/article-b': SourceReliability.MEDIUM,
    'https://apnews.com/article-c': SourceReliability.MEDIUM,
    'https://nytimes.com/article-d': SourceReliability.MEDIUM,
}
majority_record = resolve_conflict(
    claim='Hybrid work arrangements increase employee retention',
    sources_for=[
        'https://reuters.com/article-a',
        'https://bbc.com/article-b',
        'https://apnews.com/article-c',
    ],
    sources_against=['https://nytimes.com/article-d'],
    reliability_lookup=reliability_majority,
)
print(f'Resolution: {majority_record.resolution}')
print(f'Winning side: {majority_record.winning_side}')
print(f'Confidence: {majority_record.confidence:.2f}  # 3 of 4 sources agree -> 0.75')
print(f'Sources for: {len(majority_record.sources_for)}, against: {len(majority_record.sources_against)}')

## Strategy 3: Equal Reliability + Equal Count -> Human Review

In [ ]:
# Equal reliability + equal count -> human review
reliability_equal = {
    'https://reuters.com': SourceReliability.MEDIUM,
    'https://bbc.com': SourceReliability.MEDIUM,
}
record2 = resolve_conflict(
    claim='Remote work productivity',
    sources_for=['https://reuters.com'],
    sources_against=['https://bbc.com'],
    reliability_lookup=reliability_equal,
)
print(f'Resolution: {record2.resolution}')
print(f'Winning side: {record2.winning_side}')
print(f'Confidence: {record2.confidence:.2f}')

In [ ]:
from helpers import compare_results

compare_results(
    {'verdict_when_blog_reports_first': anti_a,
     'verdict_when_gov_reports_first': anti_b,
     'order_independent': anti_a == anti_b},
    {'verdict_when_blog_reports_first': correct_a,
     'verdict_when_gov_reports_first': correct_b,
     'order_independent': correct_a == correct_b},
)

### How This Connects to the ResearchReport

The `build_research_report()` function in `coordinator.py` uses conflict records
to adjust the report's overall confidence score:

```python
base_confidence = 0.8
if gaps:
    base_confidence -= 0.1 * len(gaps)     # Each gap reduces confidence
if conflict_records:
    flagged = [c for c in conflict_records
               if c.resolution == "flagged_for_human"]
    base_confidence -= 0.05 * len(flagged)  # Unresolved conflicts reduce confidence
```

Reports that say 'we could not reach Source X' or 'these sources disagree and we
flagged it for human review' are **more trustworthy** than reports that silently
omit unavailable sources or pick the first result.

## CCA Exam Tip

> The conflict resolution question tests whether you understand that:
> - Source reliability ranking is the first strategy
> - Majority consensus applies when reliability is tied
> - Human review is the fallback, not auto-resolution
> - 'First result wins' is always the wrong answer
> - The resolution is **deterministic** -- same inputs always produce same output

*Why deterministic?* Enterprise systems need **predictable, auditable,
repeatable** outcomes -- a resolution must be reviewable after the fact
and reproducible for compliance. LLM judgment cannot meet those three
requirements; programmatic rules can.